# Introduction
This notebook is used to analysis the variability of ecoli metabolite flux across the simulation. The goal is to identify the most varied metabolites and their corresponding reactions, so that we can consider them for JGI.

# Load simulation

In [10]:
import pandas as pd
import os, dill
import numpy as np
import matplotlib.pyplot as plt

from ecoli.processes.metabolism_redux_classic import NetworkFlowModel, FlowResult

os.chdir(os.path.expanduser('~/dev/vEcoli'))

%reload_ext autoreload


In [2]:
def load_sim(
        folder_path:str,
):
    """ This function is meant to load an output of a simulation in timeseries form.
        Note: This is not designed for parquet output format.
    """
    # --- Load Sim ---
    output = np.load(folder_path + '0_output.npy',allow_pickle='TRUE').item()
    output = output['agents']['0']
    fba = output['listeners']['fba_results']
    bulk = pd.DataFrame(output['bulk'])
    f = open(folder_path + 'agent_steps.pkl', 'rb')
    agent = dill.load(f)
    f.close()

    metabolism = agent['ecoli-metabolism-redux-classic']

    return fba, bulk, metabolism, output

In [3]:
time_num = 1500
date = '2026-03-24'
experiment_name = 'metabolism_classic'
condition = 'basal'

time = str(time_num)
entry = f'{experiment_name}_{time}_{date}'
folder_path = f'out_cyrus/{condition}/{entry}/'

fba, bulk, metabolism, output = load_sim(folder_path)

# Metabolite Variability Analysis -- Within Basal Simulation
<p>
For <b>non-homeostatic metabolites </b>, the dmdt will just be zero, so instead of looking at the dmdt, we will look at the fluxes of the reactions that produce or consume the metabolite (one of the direction should be sufficient for this analysis). We will look at the variability of the fluxes across the simulation, and identify the most varied metabolites and their corresponding reactions.

For <b>homeostatic metabolites </b>, the dmdt will be non-zero, for now, we can just look at the dmdt of the homeostatic metabolites and identify the most varied ones.
</p>


1. metabolite variability. heatmap ranked. metabolites on y-axis, reactions on x-axis. color = variability (std) of fluxes across simulation.
2. metabolite variability across conditions.
3. metabolite variability across time.
4. report top 20 varied metabolite

In [4]:
S = metabolism.stoichiometry.copy() # row = metabolites, col = reactions
df_S = pd.DataFrame(S, index=metabolism.metabolite_names, columns=metabolism.reaction_names)

# Split S into homeostatic and non-homeostatic metabolites
homeostatic_metabolites = metabolism.homeostatic_metabolites
df_S_nonhome = df_S.loc[~df_S.index.isin(homeostatic_metabolites), :]
df_S_home = df_S.loc[df_S.index.isin(homeostatic_metabolites), :]

# for non-homeostatic metabolites, get the reactions that produce them
mask = np.any(df_S_nonhome > 0, axis=0)
df_S_nonhome_prod = df_S_nonhome.loc[:, mask]
df_S_nonhome_prod

,1-ACYLGLYCEROL-3-P-ACYLTRANSFER-RXN,1.1.1.127-RXN,1.1.1.127-RXN (reverse),1.1.1.251-RXN,1.1.1.251-RXN (reverse),1.1.1.271-RXN (reverse),1.1.1.274-RXN (reverse),1.1.1.283-RXN (reverse),1.11.1.15-RXN,1.13.11.16-RXN,...,URUR-RXN,VAGL-RXN,VAGL-RXN (reverse),VALINE-PYRUVATE-AMINOTRANSFER-RXN,XANPRIBOSYLTRAN-RXN (reverse),XANTHOSINEPHOSPHORY-RXN,XYLISOM-RXN,XYLISOM-RXN (reverse),XYLONATE-DEHYDRATASE-RXN,YIAE1-RXN (reverse)
1-2-Diglycerides[c],0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1-AMINO-PROPAN-2-ONE-3-PHOSPHATE[c],0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1-Cys-Peroxiredoxin-L-cysteine[c],0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1-Cys-Peroxiredoxin-L-hydroxycysteine[c],0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1-KETO-2-METHYLVALERATE[c],0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
tRNAs-with-queuine[c],0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
tRNAs[c],0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
trans-delta2-arachidoyl-ACPs[c],0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
type-IV-prepillin[c],0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [5]:
# get reaction flux v to get dmdt
solution_flux = pd.DataFrame(fba["solution_fluxes"], columns=metabolism.reaction_names).loc[1:]
solution_flux_nonhome_prod = solution_flux.loc[:, df_S_nonhome_prod.columns]
df_S_nonhome_prod.dot(solution_flux_nonhome_prod)

ValueError: matrices are not aligned